# From nodal elements to hierarchical high order

Python controls the experiment, while C++ implements the basis functions, orientation and global numbering. We start with $P_1$ triangles, $Q_1$ quadrilaterals and $P_1$ tetrahedra, then compare nodal $P_2$ coordinates with hierarchical modes. `HighOrderFESpace` supports every uniform positive degree in two dimensions.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from ngsolve import *
from ngsolve.webgui import Draw
from ipywidgets import interactive, SelectionSlider
from IPython.display import display
from netgen.meshing import Mesh as NetgenMesh, MeshPoint, Pnt, Element2D, Element1D
from netgen.occ import unit_cube
from teaching_utils import load_module, square_mesh

m = load_module()

## Reference coordinates and local ordering

NGSolve orders triangle vertices as $(1,0),(0,1),(0,0)$, giving the basis $(\xi,\eta,1-\xi-\eta)$. The tetrahedron uses $(1,0,0),(0,1,0),(0,0,1),(0,0,0)$. Quadrilateral vertices are ordered cyclically as $(0,0),(1,0),(1,1),(0,1)$.

Local basis indices and the array returned by `GetDofNrs` must use the same ordering. The reference vertices are listed explicitly below. The module evaluates the basis values and gradients using the actual finite element classes.

In [ ]:
reference_vertices = {
    "triangle": [[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]],
    "quad": [[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]],
    "tetrahedron": [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0],
                    [0.0, 0.0, 1.0], [0.0, 0.0, 0.0]],
}
for kind, nodes in reference_vertices.items():
    print(f"{kind}: {len(nodes)} local degrees of freedom; nodes = {nodes}")

## Global vertex functions on triangular and quadrilateral meshes

For degree one, every global coefficient is attached to a mesh vertex. We set the coefficient at the central vertex to one and the others to zero. A triangular mesh produces piecewise affine hats, while a quadrilateral mesh produces mapped bilinear hats.

Both meshes below have the same vertices. Their local functions and cell connectivities differ.

The triangles use `P1TriangleFESpace`, whose element has explicit values and gradients. The quadrilaterals use `HighOrderFESpace` at degree one.

In [ ]:
fig = plt.figure(figsize=(11,4))
global_examples = []
xs = np.linspace(.001,.999,41)
xx, yy = np.meshgrid(xs,xs)
for panel, quad in enumerate([False,True],1):
    mesh = square_mesh(4, quadrilateral=quad)
    if quad:
        fes = m.HighOrderFESpace(mesh,order=1)
    else:
        fes = m.P1TriangleFESpace(mesh)
    coordinates = np.array([vertex.point[:2] for vertex in mesh.vertices])
    center = np.argmin(np.linalg.norm(coordinates-[.5,.5],axis=1))
    basis = GridFunction(fes)
    basis.vec[:] = 0
    basis.vec[int(center)] = 1
    zz = np.array([basis(mesh(float(a),float(b))) for a,b in zip(xx.flat,yy.flat)]).reshape(xx.shape)
    ax = fig.add_subplot(1,2,panel,projection="3d")
    ax.plot_surface(xx,yy,zz,cmap="viridis")
    ax.set(xlabel="x",ylabel="y",title="Q1 quadrilaterals" if quad else "P1 triangles")
    global_examples.append(dict(quadrilateral=quad,ndof=fes.ndof,cells=mesh.ne))
plt.tight_layout()
plt.show()

## The same construction in three dimensions

On tetrahedra there are four affine reference functions and one global degree of freedom per vertex. The boundary elements are triangles.

Solve $-\Delta u+u=f$ with $u=g$ on the entire boundary, choosing $u=x+2y+3z$, hence $f=u$ and $g=u|_{\partial\Omega}$. This affine function is represented exactly by the $P_1$ space. The plot compares the computed solution with the known function along an interior line.

In [ ]:
mesh3 = Mesh(unit_cube.GenerateMesh(maxh=.6))
fes3 = m.HighOrderFESpace(mesh3,order=1,dirichlet=".*")
u,v = fes3.TnT()
exact3 = x+2*y+3*z
a3 = BilinearForm((grad(u)*grad(v)+u*v)*dx).Assemble()
f3 = LinearForm(exact3*v*dx).Assemble()
uh3 = GridFunction(fes3)
uh3.Set(exact3,definedon=mesh3.Boundaries(".*"))
rhs3 = f3.vec.CreateVector()
rhs3.data = f3.vec-a3.mat*uh3.vec
uh3.vec.data += a3.mat.Inverse(fes3.FreeDofs(),inverse="sparsecholesky")*rhs3
print(f"Tetrahedra: {mesh3.ne}; vertex coefficients: {fes3.ndof}")
ts = np.linspace(.02,.98,25)
plt.figure(figsize=(6,3.5))
plt.plot(ts,[exact3(mesh3(float(t),.4,.6)) for t in ts],label="exact affine function")
plt.plot(ts,[uh3(mesh3(float(t),.4,.6)) for t in ts],"o",label="own P1 tetrahedra")
plt.xlabel("x at y=0.4, z=0.6")
plt.ylabel("u")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

`HighOrderFESpace` supplies the global numbering and the edge orientation used by the reference elements. In three dimensions this teaching space deliberately remains $P_1$. In two dimensions we now enrich the degree while retaining the earlier basis functions.

## Same quadratic space, different coordinates

The nodal $P_2$ basis is dual to values at vertices and edge midpoints. The hierarchical basis retains the three linear vertex functions and adds edge bubbles. These bases span the same polynomial space.

The manuscript first uses $E_{ij}=4\lambda_i\lambda_j$, with coefficient $U_{ij}=u(m_{ij})-[u(x_i)+u(x_j)]/2$. The implementation uses the integrated Legendre normalization

$$
\begin{align*}
\widetilde E_{ij}=-2\lambda_i\lambda_j=-\tfrac12 E_{ij},\qquad \widetilde U_{ij}=-2U_{ij}.
\end{align*}
$$

The vertex coefficients are unchanged. The last three entries of `hierarchical_coefficients` below therefore equal $-2$ times the manuscript's midpoint corrections, compared in the implementation's edge order. Both coefficient vectors represent the same polynomial.

A small change-of-basis calculation illustrates this relation. This dense calculation is only for the quadratic comparison, not the high-order construction algorithm.

In [ ]:
# Vertices, then midpoints of edges (2,0), (1,2), (0,1).
nodes_p2 = [
    [1.0, 0.0], [0.0, 1.0], [0.0, 0.0],
    [0.5, 0.0], [0.0, 0.5], [0.5, 0.5],
]
nodal_coefficients = np.array([a*a+b for a,b in nodes_p2])
hierarchical_at_nodes = np.array([m.HighOrderShapes("triangle",2,point)[0] for point in nodes_p2])
hierarchical_coefficients = np.linalg.solve(hierarchical_at_nodes,nodal_coefficients)
print("Nodal coefficients:",nodal_coefficients)
print("Hierarchical coefficients:",np.round(hierarchical_coefficients,6))

## Integrated Legendre edges and Dubiner bubbles

For $n\ge2$, $L_n(t)=\int_{-1}^t P_{n-1}(s)\,ds$ vanishes at both endpoints. On a triangle, homogeneous scaling extends this edge function polynomially into the cell. Interior functions are $\lambda_0\lambda_1\lambda_2 D_{ij}$, where $D_{ij}$ is the Dubiner polynomial from the chapter. Multiplication by the bubble changes its orthogonality weight.

The six panels use a single $P_4$ reference triangle: one vertex function, three modes on the same oriented edge (degrees $2,3,4$), and the interior modes $bD_{00}=b$ and $bD_{01}=b(3\lambda_2-1)$, with $b=\lambda_0\lambda_1\lambda_2$. Compare their boundary traces, the even/odd symmetry along the edge, and the interior sign change. Red and blue indicate positive and negative values. Vertical scales differ between panels; the functions retain the normalization used by the implementation. These definitions are retained at higher degrees.

In [ ]:
from matplotlib.ticker import MaxNLocator
points = np.array([(i/32, j/32) for i in range(33) for j in range(33-i)])
triangulation = mtri.Triangulation(points[:,0], points[:,1])
p = 4
hierarchical_values = np.array([m.HighOrderShapes("triangle",p,point.tolist())[0] for point in points])
first_interior = 3+3*(p-1)
mode_indices = [0,3,4,5,first_interior,first_interior+1]
titles = [r"(a) vertex $\lambda_0$",r"(b) edge mode $n=2$",
          r"(c) edge mode $n=3$",r"(d) edge mode $n=4$",
          r"(e) interior $b$",r"(f) interior $bD_{01}$"]
fig = plt.figure(figsize=(12,7.6))
for panel,(index,title) in enumerate(zip(mode_indices,titles),1):
    ax = fig.add_subplot(2,3,panel,projection="3d")
    values = hierarchical_values[:,index]
    limit = np.max(np.abs(values))
    ax.plot_trisurf(triangulation,values,cmap="RdBu_r",vmin=-limit,vmax=limit,
                    linewidth=0,antialiased=True)
    ax.plot([0,1,0,0],[0,0,1,0],[0,0,0,0],color=".35",linewidth=.8)
    ax.set_title(title,fontsize=18,pad=8)
    ax.set_xlabel(r"$\xi$",fontsize=16,labelpad=10)
    ax.set_ylabel(r"$\eta$",fontsize=16,labelpad=10)
    ax.set_xticks([0,.5,1]); ax.set_yticks([0,.5,1])
    ax.zaxis.set_major_locator(MaxNLocator(4))
    ax.tick_params(labelsize=14,pad=1)
    ax.view_init(elev=30,azim=-125)
plt.tight_layout(pad=1.7,h_pad=2.1,w_pad=1.0)
plt.show()

## Explore the basis on a single triangle

The same six functions can be viewed as finite element functions on a mesh containing just the reference triangle. Setting one coefficient to one and all others to zero selects a basis function.

Use the **Basis** slider to choose the vertex function, edge degrees 2-4, or either interior bubble. The height scale is adjusted for each mode so that small bubbles remain visible. The colorbar shows the unscaled function values.


In [ ]:
triangle_netgen = NetgenMesh(dim=2)
triangle_domain = triangle_netgen.AddRegion("domain", dim=2)
triangle_boundary = triangle_netgen.AddRegion("boundary", dim=1)
triangle_points = [triangle_netgen.Add(MeshPoint(Pnt(a, b, 0)))
                   for a, b in reference_vertices["triangle"]]
triangle_netgen.Add(Element2D(triangle_domain, triangle_points))
for a, b in [(0, 1), (1, 2), (2, 0)]:
    triangle_netgen.Add(Element1D([triangle_points[a], triangle_points[b]],
                                index=triangle_boundary))

triangle_mesh = Mesh(triangle_netgen)
triangle_order = 4
triangle_fes = m.HighOrderFESpace(triangle_mesh, order=triangle_order)
triangle_cell = next(iter(triangle_mesh.Elements()))
triangle_dofs = list(triangle_fes.GetDofNrs(triangle_cell))
triangle_first_interior = 3+3*(triangle_order-1)
triangle_modes = [0, 3, 4, 5, triangle_first_interior, triangle_first_interior+1]
triangle_scales = [0.5, 1, 2.5, 4, 12, 30]
triangle_basis = GridFunction(triangle_fes)

def draw_triangle_mode(mode):
    triangle_basis.vec[:] = 0
    triangle_basis.vec[triangle_dofs[triangle_modes[mode]]] = 1
    return Draw(triangle_basis, triangle_mesh, deformation=True,
                order=8, scale=triangle_scales[mode], euler_angles=[-60, 0, -35])

triangle_selector = SelectionSlider(
    options=[("Vertex", 0), ("Edge 2", 1), ("Edge 3", 2), ("Edge 4", 3),
             ("Bubble b", 4), ("Bubble b D01", 5)],
    description="Basis", continuous_update=False)
triangle_view = interactive(draw_triangle_mode, mode=triangle_selector)
display(triangle_view)

## Shared edge, shared coefficients

Use two triangles and degree three. Each edge has an even mode of degree two and an odd mode of degree three. The global edge direction runs from the smaller to the larger vertex number. The local-to-global map identifies the same two edge coefficients in both cells. The reference element incorporates the orientation sign into its shapes.

We set only the shared cubic edge coefficient to one. Compare the two traces at the same physical edge points. Removing the orientation sign reverses one trace. For nodal cubic elements, the corresponding operation exchanges two edge nodes instead.

The C++ space supplies global vertex numbers through `SetVertexNumbers`. NGSolve's base class `VertexOrientedFE<ET>` supplies the ordered endpoints through `GetVertexOrientedEdge`, and the basis formulas use these endpoints directly.

In [ ]:
patch = square_mesh(1)
patch_fes = m.HighOrderFESpace(patch,order=3)
patch_cells = list(patch.Elements())
shared_edge = next(iter({e.nr for e in patch_cells[0].edges}
                        & {e.nr for e in patch_cells[1].edges}))
shared_vertices = sorted({v.nr for v in patch_cells[0].vertices}
                         & {v.nr for v in patch_cells[1].vertices})
reference_triangle = np.array(reference_vertices["triangle"])
reference_edges = [(2,0),(1,2),(0,1)]
edge_parameters = np.array([0.2,0.4,0.7])
oriented_traces, raw_traces = [], []
shared_edge_dofs = []
for cell in patch_cells:
    vertices = [v.nr for v in cell.vertices]
    edge = [e.nr for e in cell.edges].index(shared_edge)
    a,b = reference_edges[edge]
    dofs = list(patch_fes.GetDofNrs(cell))
    odd_mode = 3+2*edge+1
    shared_edge_dofs.append(dofs[odd_mode])
    print(f"cell {cell.nr}: vertices {vertices}, local edge {vertices[a],vertices[b]}")
    print("  local-to-global map:",dofs)
    i,j = [vertices.index(v) for v in shared_vertices]
    points_on_edge = [(1-t)*reference_triangle[i]+t*reference_triangle[j]
                      for t in edge_parameters]
    trace = np.array([m.HighOrderShapes("triangle",3,point.tolist(),vertices)[0][odd_mode]
                      for point in points_on_edge])
    orientation = 1 if vertices[a]<vertices[b] else -1
    oriented_traces.append(trace)
    raw_traces.append(orientation*trace)
print("Shared cubic coefficient numbers:",shared_edge_dofs)
print("Traces without orientation signs:",np.round(raw_traces,6))
print("Traces with orientation signs:",np.round(oriented_traces,6))
shared_trace_difference = np.max(np.abs(oriented_traces[0]-oriented_traces[1]))
print(f"Maximum trace difference: {shared_trace_difference:.3e}")

## A shared edge between a triangle and a quadrilateral

Now join a $P_4$ triangle to a $Q_4$ quadrilateral along the vertical edge $x=1$. Both elements use the same three global coefficients on this edge. For each mode, we set one of these coefficients to one and all other coefficients to zero.

The **Degree** slider switches between edge degrees 2, 3 and 4. Rotate the view to inspect the common edge: the function values agree, although the extensions into the two cells differ. Their gradients may jump. The cubic mode also shows why both cells must use the same edge orientation.


In [ ]:
mixed_netgen = NetgenMesh(dim=2)
mixed_domain = mixed_netgen.AddRegion("domain", dim=2)
mixed_boundary = mixed_netgen.AddRegion("boundary", dim=1)
mixed_coordinates = [(0, 0), (1, 0), (1, 1), (2, 0), (2, 1)]
mixed_points = [mixed_netgen.Add(MeshPoint(Pnt(a, b, 0)))
                for a, b in mixed_coordinates]
for vertices in [(0, 1, 2), (1, 3, 4, 2)]:
    mixed_netgen.Add(Element2D(mixed_domain, [mixed_points[i] for i in vertices]))
for a, b in [(0, 1), (1, 3), (3, 4), (4, 2), (2, 0)]:
    mixed_netgen.Add(Element1D([mixed_points[a], mixed_points[b]],
                              index=mixed_boundary))

mixed_mesh = Mesh(mixed_netgen)
mixed_fes = m.HighOrderFESpace(mixed_mesh, order=4)
mixed_cells = list(mixed_mesh.Elements())
mixed_shared_edge = next(iter({edge.nr for edge in mixed_cells[0].edges}
                              & {edge.nr for edge in mixed_cells[1].edges}))
mixed_shared_dofs = list(mixed_fes.GetDofNrs(NodeId(EDGE, mixed_shared_edge)))
mixed_basis = GridFunction(mixed_fes)
mixed_scales = [1, 2.5, 4]

def draw_mixed_mode(mode):
    mixed_basis.vec[:] = 0
    mixed_basis.vec[mixed_shared_dofs[mode]] = 1
    return Draw(mixed_basis, mixed_mesh, deformation=True,
                order=8, scale=mixed_scales[mode], euler_angles=[-60, 0, -35])

print("Shared edge coefficients, in degree order 2, 3, 4:", mixed_shared_dofs)
mixed_selector = SelectionSlider(
    options=[("Edge 2", 0), ("Edge 3", 1), ("Edge 4", 2)], value=1,
    description="Degree", continuous_update=False)
mixed_view = interactive(draw_mixed_mode, mode=mixed_selector)
display(mixed_view)

## Increasing the degree on a fixed mesh

Keep the mesh fixed and solve $-\Delta u=f$ with

$$
\begin{align*}
u=e^{2x+y}\sin(\pi x)\sin(\pi y),\qquad f=-\Delta u,
\end{align*}
$$

and homogeneous Dirichlet data. The source is computed by differentiating the prescribed exact function. Triangles use $P_p$, quadrilaterals use mapped $Q_p$.

For each degree, compare our computed **function** with a separately assembled native `H1` solution. Quadrature order is increased with the degree.


In [ ]:
from teaching_utils import errors
smooth = exp(2*x+y)*sin(pi*x)*sin(pi*y)
smooth_source = -smooth.Diff(x).Diff(x)-smooth.Diff(y).Diff(y)
p_convergence = []
for quad in [False,True]:
    mesh_p = square_mesh(2,quadrilateral=quad)
    for degree in range(1,7):
        space_p = m.HighOrderFESpace(mesh_p,order=degree,dirichlet=".*")
        q = 2*degree+10
        mat_p = m.AssembleMatrix(space_p,[m.Laplace(CF(1),quadrature_order=q)])
        rhs_p = m.AssembleVector(space_p,[m.Source(smooth_source,quadrature_order=q)])
        own_p = GridFunction(space_p)
        own_p.vec.data = mat_p.Inverse(space_p.FreeDofs(),inverse="sparsecholesky")*rhs_p
        native_p = H1(mesh_p,order=degree,dirichlet=".*")
        u,v = native_p.TnT()
        an = BilinearForm(grad(u)*grad(v)*dx(bonus_intorder=q)).Assemble()
        fn = LinearForm(smooth_source*v*dx(bonus_intorder=q)).Assemble()
        field_p = GridFunction(native_p)
        field_p.vec.data = an.mat.Inverse(native_p.FreeDofs(),inverse="sparsecholesky")*fn.vec
        p_convergence.append(dict(quad=quad,p=degree,ndof=space_p.ndof,
            native_difference=sqrt(Integrate((own_p-field_p)**2,mesh_p,order=q)),
            **errors(mesh_p,own_p,smooth,quadrature_order=q)))
plt.figure(figsize=(7,4))
for quad in [False,True]:
    rows = [row for row in p_convergence if row["quad"]==quad]
    for norm,marker in [("H1","o-"),("L2","s--")]:
        plt.semilogy([r["p"] for r in rows],[r[norm] for r in rows],marker,
                     label=f"{norm}, {'Qp' if quad else 'Pp'}")
plt.xlabel("uniform degree p on a fixed mesh"); plt.ylabel("actual error")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
print("Largest native-field difference:",max(r["native_difference"] for r in p_convergence))